# Completion — checking, deriving, defaulting, and asking

This walks through the pipeline stage between import and calibration: generate a hypothetical project, import it, see which fields `core/known_fields.py` expects and which of them are actually present, run completion's automatic pass, and answer what's left by hand (standing in for a future GUI).

In [1]:
import sys
from pathlib import Path

# walk upward from this notebook until we find the project root
# (identified by the presence of the data_import folder)
here = Path.cwd()
project_root = next(p for p in [here, *here.parents] if (p / "data_import").is_dir())
sys.path.insert(0, str(project_root))

print("Using project root:", project_root)

Using project root: D:\GitHub\Universial-Reconstructions - Copy\.claude\worktrees\code-package-review-13feda


## 1. Generate a hypothetical project

Using the same fixture generator the test suite uses (`tests/fixtures/generate_hypothetical_synchrotron.py`), so this notebook doesn't depend on a file being generated ahead of time -- it makes its own.

In [2]:
from tests.fixtures.generate_hypothetical_synchrotron import generate

out_path = project_root / "doc/demo/generated/completion_demo.h5"

volume, projections, flats, angles = generate(
    out_path, shape=(48, 48, 64), n_angles=30, n_flats=5, seed=1
)
print("Wrote", out_path)
print("projections shape:", projections.shape)
print("angle span:", angles.min(), "-", angles.max())

Wrote D:\GitHub\Universial-Reconstructions - Copy\.claude\worktrees\code-package-review-13feda\doc\demo\generated\completion_demo.h5
projections shape: (30, 48, 64)
angle span: 0.0 - 174.0


## 2. Import it

Same as any other import -- `DataImporter` auto-detects the vendor plugin from the file's signature; we only have to say the format.

In [3]:
from data_import.importer import DataImporter
from core.data_interface import DataInterface

importer = DataImporter()
project = importer.import_project(out_path, format="hdf5", storage_mode="ram")
print(project)

ProjectObject(format=hdf5, storage_mode=ram, vendor=hypothetical_synchrotron_hdf5, handles=['projections', 'flats'])


## 3. Check which fields are there, and which aren't

`core.known_fields.status()` reads `core/known_fields.py`'s catalog against this project's `MetadataBundle` and reports every field the software depends on -- present or not -- regardless of whether the vendor plugin ever mentioned it.

In [4]:
from core.known_fields import status


def show_status(project):
    rows = status(project.metadata)
    width = max(len(r.key) for r in rows)
    for r in rows:
        mark = "OK" if r.resolved else "--"
        strategies = "/".join(r.strategies) or "(none)"
        value = "" if not r.resolved else f"= {r.value!r}"
        print(f"[{mark}] {r.key:<{width}}  tier={r.tier:<14} strategy={strategies:<16} {value}")


show_status(project)

[OK] angles                       tier=mandatory_soft strategy=derive/ask       = array([  0.,   6.,  12.,  18.,  24.,  30.,  36.,  42.,  48.,  54.,  60.,
        66.,  72.,  78.,  84.,  90.,  96., 102., 108., 114., 120., 126.,
       132., 138., 144., 150., 156., 162., 168., 174.])
[--] geometry_type                tier=mandatory_soft strategy=ask              
[--] angle_span                   tier=mandatory_soft strategy=derive/ask       
[--] rotation_direction           tier=optional       strategy=default          
[OK] voxel_size                   tier=mandatory_soft strategy=default          = np.float64(1.1)
[--] detector_pixel_size          tier=mandatory_soft strategy=derive/default   
[--] source_to_detector_distance  tier=mandatory_soft strategy=ask              
[--] source_to_object_distance    tier=mandatory_soft strategy=ask              
[--] center_of_rotation           tier=computed       strategy=calculate        
[--] tilt_angle                   tier=computed    

A few things to notice already:

- `angles` and `voxel_size` came straight from the file -- this vendor provides both.
- `geometry_type` and `angle_span` are missing -- this vendor's files never say either.
- `source_to_detector_distance` / `source_to_object_distance` aren't listed at all yet. They're cone-beam-only (`condition=("geometry_type", "cone")` in the registry), and since `geometry_type` isn't resolved yet, `reconcile()` correctly doesn't flag them -- see the bonus section below.
- `center_of_rotation` / `tilt_angle` are `computed`-tier: never blocking, and untouched by anything so far.

## 4. Run completion's automatic pass

`completion.run()` tries `derive` then `default`, in order, for every missing field, and stops there -- it never asks anyone anything.

In [5]:
import completion

completion.run(project)
show_status(project)

[OK] angles                       tier=mandatory_soft strategy=derive/ask       = array([  0.,   6.,  12.,  18.,  24.,  30.,  36.,  42.,  48.,  54.,  60.,
        66.,  72.,  78.,  84.,  90.,  96., 102., 108., 114., 120., 126.,
       132., 138., 144., 150., 156., 162., 168., 174.])
[--] geometry_type                tier=mandatory_soft strategy=ask              
[OK] angle_span                   tier=mandatory_soft strategy=derive/ask       = 174.0
[OK] rotation_direction           tier=optional       strategy=default          = 'ccw'
[OK] voxel_size                   tier=mandatory_soft strategy=default          = np.float64(1.1)
[OK] detector_pixel_size          tier=mandatory_soft strategy=derive/default   = np.float64(1.1)
[--] source_to_detector_distance  tier=mandatory_soft strategy=ask              
[--] source_to_object_distance    tier=mandatory_soft strategy=ask              
[--] center_of_rotation           tier=computed       strategy=calculate        
[--] tilt_angle     

`voxel_size` and `detector_pixel_size` are resolved with `source="default"`/`source="derived"` respectively (`detector_pixel_size` derives from `voxel_size` -- see `completion/derivers.py`). `angle_span` is also resolved now, `source="derived"` -- this vendor already provides `angles` directly, and `angle_span`'s deriver just computes `max(angles) - min(angles)` from it. Only `geometry_type` is still missing: it has no `derive` or `default` strategy at all, just `ask` -- there's genuinely nothing to work it out from automatically.

## 5. What's left — asking the user

`project.needs_user_input()` / `missing_soft_fields()` are the same readiness checks that already live on `core/project.py` -- completion doesn't invent a separate one. In a notebook, "asking" means calling `completion.provide()` directly; a future GUI would make the same call from a form instead.

In [6]:
print("needs_user_input:", project.needs_user_input())
print("missing_soft_fields:", project.missing_soft_fields())

needs_user_input: True
missing_soft_fields: ['geometry_type']


In [7]:
# Standing in for whatever collected this answer from a person --
# a notebook cell today, a form later. This project really is
# parallel-beam (generate_hypothetical_synchrotron.py projects with
# project_parallel).
completion.provide(project, "geometry_type", "parallel")

show_status(project)

[OK] angles                       tier=mandatory_soft strategy=derive/ask       = array([  0.,   6.,  12.,  18.,  24.,  30.,  36.,  42.,  48.,  54.,  60.,
        66.,  72.,  78.,  84.,  90.,  96., 102., 108., 114., 120., 126.,
       132., 138., 144., 150., 156., 162., 168., 174.])
[OK] geometry_type                tier=mandatory_soft strategy=ask              = 'parallel'
[OK] angle_span                   tier=mandatory_soft strategy=derive/ask       = 174.0
[OK] rotation_direction           tier=optional       strategy=default          = 'ccw'
[OK] voxel_size                   tier=mandatory_soft strategy=default          = np.float64(1.1)
[OK] detector_pixel_size          tier=mandatory_soft strategy=derive/default   = np.float64(1.1)
[--] source_to_detector_distance  tier=mandatory_soft strategy=ask              
[--] source_to_object_distance    tier=mandatory_soft strategy=ask              
[--] center_of_rotation           tier=computed       strategy=calculate        
[--] til

In [8]:
print("needs_user_input:", project.needs_user_input())

needs_user_input: False


Notice `source_to_detector_distance` / `source_to_object_distance` never appeared, because this project answered `geometry_type="parallel"`, not `"cone"` -- their `condition` was never satisfied. Quick illustration of what happens on a cone-beam project instead, using a throwaway bundle so it doesn't touch the real project above:

In [9]:
from core.metadata import MetadataBundle
from core.known_fields import reconcile

demo_bundle = MetadataBundle()
demo_bundle.declare_missing("geometry_type", tier="mandatory_soft")
reconcile(demo_bundle)
print("cone-only field before answering geometry_type:", demo_bundle.get_field("source_to_detector_distance"))

demo_bundle.set("geometry_type", "cone", tier="mandatory_soft", source="user")
reconcile(demo_bundle)
print("cone-only field after answering geometry_type='cone':", demo_bundle.get_field("source_to_detector_distance"))

cone-only field before answering geometry_type: None
cone-only field after answering geometry_type='cone': FieldSpec(key='source_to_detector_distance', tier='mandatory_soft', value=None, source=None)


This vendor already provides `angles` directly, so the demo above only ever exercised the `angle_span`-from-`angles` direction. The reverse -- `angles` from `angle_span` (plus projection count and `rotation_direction`) -- is the more interesting derive, so here it is on a throwaway project with only `angle_span` given:

In [10]:
import numpy as np

from core.handles import RamHandle
from core.project import ProjectObject

demo_meta = MetadataBundle()
demo_meta.set("projections_path", "/projections", tier="mandatory_hard")
demo_meta.set("angle_span", 180.0, tier="mandatory_soft", source="user")
demo_project = ProjectObject(
    format="hdf5", storage_mode="ram", metadata=demo_meta,
    data_handles={"projections": RamHandle(np.zeros((6, 1, 1), dtype=np.float32))},
)

completion.run(demo_project)
print("derived angles:", demo_project.metadata.get("angles"))
print("source:", demo_project.metadata.get_field("angles").source)

derived angles: [  0.  30.  60.  90. 120. 150.]
source: derived


## 6. Do we have any actual calculation yet?

Split answer -- one form exists now, the other still doesn't:

- **`derive`**: yes, now. `angle_span`, `angles`, and `detector_pixel_size` all have real derive functions (`completion/derivers.py`), registered automatically just by importing `completion`.
- **`calculate`** (calibration -- center of rotation, tilt angle): still not built at all. `completion.run()` deliberately never touches `calculate`-strategy fields, so they stay unresolved even after a project is otherwise fully completed:

In [11]:
from completion.engine import _DERIVERS

print("registered derivers:", _DERIVERS)

for key in ("center_of_rotation", "tilt_angle"):
    field = project.metadata.get_field(key)
    print(f"{key}: is_missing={field.is_missing}, value={field.value}")

registered derivers: {'angle_span': <function derive_angle_span at 0x000001E27F8E90C0>, 'angles': <function derive_angles at 0x000001E27F8E9220>, 'detector_pixel_size': <function derive_detector_pixel_size at 0x000001E27F8E9380>}
center_of_rotation: is_missing=True, value=None
tilt_angle: is_missing=True, value=None


One honest gap this still surfaces: `core/known_fields.py` records `default=0.0` for both fields, as the seed guess calibration is meant to start from -- but that default is never actually applied, because `completion.run()` only ever applies the `default` strategy, and these two fields' only strategy is `calculate`. Right now that `default=0.0` is just documentation of calibration's intended starting point, not a live behavior. Worth deciding when calibration gets built: should calibration seed itself, or should completion apply computed-tier defaults too? Left open on purpose.

## Recap

```
import -> completion.run() (automatic) -> completion.provide() x N (manual) -> project.needs_user_input() == False
```

This project is now completion-ready. Calibration (`center_of_rotation`, `tilt_angle`) is the next stage -- not built yet.

In [12]:
assert not project.needs_user_input()
print(project)

ProjectObject(format=hdf5, storage_mode=ram, vendor=hypothetical_synchrotron_hdf5, handles=['projections', 'flats'])
